# Test Widget - Debugging Duplicate Rendering

This notebook creates a minimal test case to debug the duplicate rendering issue.
We'll test with the simplest possible widget: one button, one output area.

In [1]:
# Import the test widget
from test_widget import TestWidget, create_test_widget
import ipywidgets as widgets
from IPython.display import display, clear_output

## Test 1: Basic Minimal Widget

In [2]:
# Create and show the minimal test widget
test_widget = create_test_widget()
test_widget.show()

print("Test widget created. Click the button and watch:")
print("1. The debug message in this cell output")
print("2. The counter in the widget output area")
print("If there's duplicate rendering, you'll see multiple debug messages per click.")

Test widget created. Click the button and watch:
1. The debug message in this cell output
2. The counter in the widget output area
If there's duplicate rendering, you'll see multiple debug messages per click.


## Test 2: Even Simpler - Direct Button Test

In [3]:
# Create the simplest possible test - direct button with lambda
simple_counter = [0]  # Use list for mutable counter

def simple_click_handler(button):
    simple_counter[0] += 1
    print(f"SIMPLE DEBUG: Click #{simple_counter[0]}")
    with simple_output:
        clear_output(wait=True)
        print(f"Simple click count: {simple_counter[0]}")

simple_button = widgets.Button(description='Simple Test')
simple_output = widgets.Output()

simple_button.on_click(simple_click_handler)

display(widgets.VBox([
    widgets.HTML("<h4>Ultra-Simple Test</h4>"),
    simple_button,
    simple_output
]))

print("Ultra-simple widget created. Watch for duplicate debug messages.")

Ultra-simple widget created. Watch for duplicate debug messages.


## Test 3: Check Event Handler Attachment

In [5]:
# Check if multiple handlers are accidentally attached (ipywidgets 8.1.5 compatible)
print("Test Widget Button Handlers:")
test_dispatcher = test_widget.test_button._click_handlers
print(f"Click dispatcher type: {type(test_dispatcher)}")

# Try different ways to inspect the CallbackDispatcher
attrs_to_check = ['callbacks', '_callbacks', '_callback_list', 'registered_callbacks']
print(f"Available attributes: {[attr for attr in attrs_to_check if hasattr(test_dispatcher, attr)]}")

print("\nSimple Button Handlers:")  
simple_dispatcher = simple_button._click_handlers
print(f"Click dispatcher type: {type(simple_dispatcher)}")
print(f"Available attributes: {[attr for attr in attrs_to_check if hasattr(simple_dispatcher, attr)]}")

# Try to get callback count using available methods
def get_callback_info(dispatcher, name):
    print(f"\n=== {name} Callback Analysis ===")
    try:
        # Check common attributes that might hold callbacks
        if hasattr(dispatcher, 'callbacks'):
            callbacks = dispatcher.callbacks
            print(f"callbacks attribute: {len(callbacks)} items")
            for i, cb in enumerate(callbacks):
                print(f"  [{i}] {cb}")
        elif hasattr(dispatcher, '_callbacks'):
            callbacks = dispatcher._callbacks  
            print(f"_callbacks attribute: {len(callbacks)} items")
            for i, cb in enumerate(callbacks):
                print(f"  [{i}] {cb}")
        else:
            # Try to inspect all attributes
            all_attrs = [attr for attr in dir(dispatcher) if not attr.startswith('__')]
            print(f"All non-dunder attributes: {all_attrs}")
            
            # Look for list-like or dict-like attributes
            for attr in all_attrs:
                try:
                    value = getattr(dispatcher, attr)
                    if hasattr(value, '__len__') and not callable(value):
                        print(f"  {attr}: {type(value)} with length {len(value)}")
                        if len(value) > 0:
                            print(f"    Contents: {value}")
                except:
                    print(f"  {attr}: {type(value)} (couldn't get length)")
                    
    except Exception as e:
        print(f"Error inspecting {name}: {e}")

get_callback_info(test_dispatcher, "Test Widget")
get_callback_info(simple_dispatcher, "Simple Widget")

Test Widget Button Handlers:
Click dispatcher type: <class 'ipywidgets.widgets.widget.CallbackDispatcher'>
Available attributes: ['callbacks']

Simple Button Handlers:
Click dispatcher type: <class 'ipywidgets.widgets.widget.CallbackDispatcher'>
Available attributes: ['callbacks']

=== Test Widget Callback Analysis ===
callbacks attribute: 1 items
  [0] <bound method TestWidget._on_button_click of <test_widget.TestWidget object at 0x7e860e28e120>>

=== Simple Widget Callback Analysis ===
callbacks attribute: 1 items
  [0] <function simple_click_handler at 0x7e860e1cd260>


## Test 4: Trace Method Calls

In [7]:
# Create a traced version to see exactly what's happening
import time

class TracedTestWidget(TestWidget):
    def __init__(self):
        super().__init__()
        self.call_log = []
    
    def _on_button_click(self, button):
        call_time = time.time()
        self.call_log.append(call_time)
        
        print(f"TRACE: _on_button_click called at {call_time}")
        print(f"TRACE: Total calls so far: {len(self.call_log)}")
        
        # Check for rapid duplicate calls (within 100ms)
        if len(self.call_log) > 1:
            time_diff = call_time - self.call_log[-2]
            if time_diff < 0.1:  # Less than 100ms
                print(f"TRACE: ⚠️ RAPID DUPLICATE CALL! Only {time_diff:.3f}s since last call")
            else:
                print(f"TRACE: Normal call - {time_diff:.3f}s since last call")
        
        super()._on_button_click(button)

# Create traced widget
traced_widget = TracedTestWidget()
traced_widget.show()

print("Traced widget created. This will show detailed timing information.")

Traced widget created. This will show detailed timing information.


## Analysis Cell

In [8]:
# Analyze the results
print("=== ANALYSIS ===\n")

if 'test_widget' in locals():
    print(f"Test Widget Click Count: {test_widget.click_count}")

if 'simple_counter' in locals():
    print(f"Simple Counter Value: {simple_counter[0]}")

if 'traced_widget' in locals():
    print(f"Traced Widget Click Count: {traced_widget.click_count}")
    print(f"Traced Widget Call Log Length: {len(traced_widget.call_log)}")
    
    if len(traced_widget.call_log) > 1:
        print("\nCall Timing Analysis:")
        for i in range(1, len(traced_widget.call_log)):
            time_diff = traced_widget.call_log[i] - traced_widget.call_log[i-1]
            print(f"  Call {i-1} -> {i}: {time_diff:.3f}s")
            if time_diff < 0.1:
                print(f"    ⚠️ SUSPICIOUS: Very rapid call!")

print("\n=== CONCLUSIONS ===\n")
print("Look for:")
print("1. Multiple debug messages per click (indicates duplicate calls)")
print("2. Rapid successive calls in trace analysis (< 100ms apart)")
print("3. Multiple handlers attached to buttons")
print("4. Click count vs call log mismatches")

=== ANALYSIS ===

Test Widget Click Count: 0
Simple Counter Value: 0
Traced Widget Click Count: 0
Traced Widget Call Log Length: 0

=== CONCLUSIONS ===

Look for:
1. Multiple debug messages per click (indicates duplicate calls)
2. Rapid successive calls in trace analysis (< 100ms apart)
3. Multiple handlers attached to buttons
4. Click count vs call log mismatches
